# OneStep PCBM – Huấn luyện mô hình AI nhận thức cá nhân
Huấn luyện hai mô hình:
   - **Mô hình dự đoán khả năng tự nhớ**;
   - **Mô hình dự đoán khả năng hoàn thành trong 30 phút**.


## Cài thư viện


In [ ]:
# Cài thư viện nếu môi trường chưa có
# !pip install pandas numpy scikit-learn matplotlib joblib

In [ ]:
from pathlib import Path
import json
import math
from datetime import timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
import joblib

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

THU_MUC_DU_AN = Path("/content/drive/MyDrive/ADHD OneStep/training model")
TEP_DU_LIEU = THU_MUC_DU_AN / "sample_events.jsonl"

In [ ]:
print(TEP_DU_LIEU)

## Đọc dữ liệu sự kiện

In [ ]:
su_kien = []

with TEP_DU_LIEU.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        e = json.loads(line)
        e["timestamp"] = pd.Timestamp(e["timestamp"])
        e["event_type"] = e["event_type"].upper().strip()
        su_kien.append(e)

su_kien = sorted(su_kien, key=lambda x: (x["user_id"], x["timestamp"], x["event_id"]))

print("Tổng số sự kiện:", len(su_kien))
pd.DataFrame([
    {
        "event_id": e["event_id"],
        "user_id": e["user_id"],
        "task_id": e["task_id"],
        "occurrence_id": e["occurrence_id"],
        "timestamp": e["timestamp"],
        "event_type": e["event_type"],
    }
    for e in su_kien
]).head(12)

## Quan sát phân bố loại sự kiện

In [ ]:
bang_su_kien = pd.DataFrame([{
    "user_id": e["user_id"],
    "event_type": e["event_type"],
    "timestamp": e["timestamp"]
} for e in su_kien])

so_luong_theo_loai = bang_su_kien["event_type"].value_counts().sort_values(ascending=False)
display(so_luong_theo_loai.to_frame("count"))

plt.figure(figsize=(10, 4))
so_luong_theo_loai.plot(kind="bar")
plt.title("Số lượng sự kiện theo loại")
plt.ylabel("Số sự kiện")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Xây dựng đặc trưng

Nguyên tắc quan trọng:

> Tại thời điểm ra quyết định `t`, chỉ được dùng dữ liệu có `timestamp < t`.

Điều này tránh **rò rỉ dữ liệu** từ tương lai vào đặc trưng huấn luyện.

In [ ]:
def tinh_ty_le_an_toan(num, den):
    return num / den if den else np.nan

def xac_dinh_buoi_trong_ngay(hour):
    if 5 <= hour < 12:
        return "buổi_sáng"
    if 12 <= hour < 17:
        return "buổi_chiều"
    if 17 <= hour < 22:
        return "buổi_tối"
    return "ban_đêm"

def lay_tap_lan_thuc_hien(history, event_type_set):
    return {
        e["occurrence_id"]
        for e in history
        if e["event_type"] in event_type_set
    }

def lay_su_kien_tao_nhiem_vu_truoc(su_kien, occurrence_id, decision_time):
    xs = [
        e for e in su_kien
        if e["occurrence_id"] == occurrence_id
        and e["event_type"] == "TASK_CREATED"
        and e["timestamp"] < decision_time
    ]
    return xs[-1] if xs else None

def xay_dung_dac_trung(su_kien, decision):
    t = decision["timestamp"]
    user_id = decision["user_id"]
    task_id = decision["task_id"]
    occ = decision["occurrence_id"]

    # Chỉ sử dụng các sự kiện xảy ra trước thời điểm ra quyết định
    history = [
        e for e in su_kien
        if e["user_id"] == user_id and e["timestamp"] < t
    ]
    hist_7d = [
        e for e in history
        if e["timestamp"] >= t - pd.Timedelta(days=7)
    ]

    created = lay_su_kien_tao_nhiem_vu_truoc(su_kien, occ, t)
    task_data = (created or {}).get("data", {})
    reminder_data = decision.get("data", {})

    reminded_7d = lay_tap_lan_thuc_hien(hist_7d, {"REMINDER_SENT"})
    completed_7d = lay_tap_lan_thuc_hien(hist_7d, {"TASK_COMPLETED"})
    snoozed_7d = lay_tap_lan_thuc_hien(hist_7d, {"SNOOZE"})
    forgot_7d = lay_tap_lan_thuc_hien(hist_7d, {"USER_FORGOT", "RECALL_FAILURE"})

    same_task_hist = [e for e in history if e["task_id"] == task_id]
    same_task_reminded = lay_tap_lan_thuc_hien(same_task_hist, {"REMINDER_SENT"})
    same_task_completed = lay_tap_lan_thuc_hien(same_task_hist, {"TASK_COMPLETED"})

    # Tính độ trễ bắt đầu trong 7 ngày gần nhất
    reminder_times = {}
    start_times = {}

    for e in hist_7d:
        if e["event_type"] == "REMINDER_SENT":
            reminder_times[e["occurrence_id"]] = e["timestamp"]
        elif e["event_type"] == "TASK_STARTED":
            start_times[e["occurrence_id"]] = e["timestamp"]

    delays = []
    for oid, rt in reminder_times.items():
        st = start_times.get(oid)
        if st is not None and st >= rt:
            delays.append((st - rt).total_seconds() / 60)

    first_hist_time = min(
        (e["timestamp"] for e in history),
        default=t
    )

    hour = t.hour
    weekday = t.weekday()

    return {
        "user_id": user_id,
        "task_id": task_id,
        "occurrence_id": occ,
        "decision_time": t,

        "hour": hour,
        "weekday": weekday,
        "is_weekend": int(weekday >= 5),
        "xac_dinh_buoi_trong_ngay": xac_dinh_buoi_trong_ngay(hour),

        # Mã hóa chu kỳ thời gian bằng sin/cos
        "hour_sin": math.sin(2 * math.pi * hour / 24),
        "hour_cos": math.cos(2 * math.pi * hour / 24),

        # Thông tin nhiệm vụ
        "task_type": task_data.get("task_type", "UNKNOWN"),
        "priority": task_data.get("priority"),
        "estimated_minutes": task_data.get("estimated_minutes"),

        # Mức hỗ trợ/can thiệp hiện tại
        "prompt_level": reminder_data.get("prompt_level"),
        "reminder_type": reminder_data.get("reminder_type", "UNKNOWN"),
        "channel": reminder_data.get("channel", "UNKNOWN"),

        # Đặc trưng lịch sử cá nhân
        "completion_rate_7d": tinh_ty_le_an_toan(
            len(completed_7d & reminded_7d),
            len(reminded_7d)
        ),
        "snooze_rate_7d": tinh_ty_le_an_toan(
            len(snoozed_7d & reminded_7d),
            len(reminded_7d)
        ),
        "forget_rate_7d": tinh_ty_le_an_toan(
            len(forgot_7d & reminded_7d),
            len(reminded_7d)
        ),
        "avg_start_delay_7d": (
            np.mean(delays) if delays else np.nan
        ),

        "same_task_completion_rate": tinh_ty_le_an_toan(
            len(same_task_completed & same_task_reminded),
            len(same_task_reminded)
        ),

        "event_count_before": len(history),
        "history_days": max(
            0.0,
            (t - first_hist_time).total_seconds() / 86400
        ),
        "is_cold_start": int(len(history) < 10),
    }

## Sinh nhãn huấn luyện

Nhãn được lấy từ các sự kiện **sau** thời điểm ra quyết định.

Quy tắc demo:

- Tự nhớ: trong 2 phút.
- Bắt đầu: trong 10 phút.
- Hoàn thành: trong 30 phút và 60 phút.
- Trì hoãn: trong 10 phút.
- Quên: chỉ dùng kết quả có xác nhận rõ ràng.

Nếu không có đủ bằng chứng cho nhãn `recall` hoặc `forgot`, giữ `NaN` thay vì tự động gán thất bại.

In [ ]:
def lay_su_kien_tuong_lai_dau_tien(su_kien, occurrence_id, event_types, start, end):
    xs = [
        e for e in su_kien
        if e["occurrence_id"] == occurrence_id
        and e["event_type"] in event_types
        and start < e["timestamp"] <= end
    ]
    xs.sort(key=lambda x: x["timestamp"])
    return xs[0] if xs else None

def xay_dung_nhan(su_kien, decision):
    t = decision["timestamp"]
    occ = decision["occurrence_id"]

    out = {
        "y_recall_success": np.nan,
        "y_started_within_10m": 0,
        "y_start_delay_minutes": np.nan,
        "y_complete_within_30m": 0,
        "y_complete_within_60m": 0,
        "y_completion_delay_minutes": np.nan,
        "y_snooze": 0,
        "y_forgot": np.nan,
    }

    recall = lay_su_kien_tuong_lai_dau_tien(
        su_kien, occ,
        {"RECALL_SUCCESS", "RECALL_FAILURE"},
        t, t + pd.Timedelta(minutes=2)
    )
    if recall:
        out["y_recall_success"] = int(
            recall["event_type"] == "RECALL_SUCCESS"
        )

    started = lay_su_kien_tuong_lai_dau_tien(
        su_kien, occ,
        {"TASK_STARTED"},
        t, t + pd.Timedelta(minutes=60)
    )
    if started:
        delay = (started["timestamp"] - t).total_seconds() / 60
        out["y_start_delay_minutes"] = delay
        out["y_started_within_10m"] = int(delay <= 10)

    completed = lay_su_kien_tuong_lai_dau_tien(
        su_kien, occ,
        {"TASK_COMPLETED"},
        t, t + pd.Timedelta(minutes=180)
    )
    if completed:
        delay = (completed["timestamp"] - t).total_seconds() / 60
        out["y_completion_delay_minutes"] = delay
        out["y_complete_within_30m"] = int(delay <= 30)
        out["y_complete_within_60m"] = int(delay <= 60)

    snooze = lay_su_kien_tuong_lai_dau_tien(
        su_kien, occ,
        {"SNOOZE"},
        t, t + pd.Timedelta(minutes=10)
    )
    out["y_snooze"] = int(snooze is not None)

    forgot = lay_su_kien_tuong_lai_dau_tien(
        su_kien, occ,
        {"USER_FORGOT", "USER_REMEMBERED",
         "RECALL_FAILURE", "RECALL_SUCCESS"},
        t, t + pd.Timedelta(minutes=10)
    )
    if forgot:
        if forgot["event_type"] in {"USER_FORGOT", "RECALL_FAILURE"}:
            out["y_forgot"] = 1
        else:
            out["y_forgot"] = 0

    return out

## Tạo bảng dữ liệu huấn luyện

In [ ]:
cac_diem_quyet_dinh = [
    e for e in su_kien
    if e["event_type"] == "REMINDER_SENT"
]

cac_ban_ghi_hoc = []

for d in cac_diem_quyet_dinh:
    row = {}
    row.update(xay_dung_dac_trung(su_kien, d))
    row.update(xay_dung_nhan(su_kien, d))
    cac_ban_ghi_hoc.append(row)

df = pd.DataFrame(cac_ban_ghi_hoc)

print("Số điểm ra quyết định:", len(df))
display(df.head())

## Kiểm tra nhanh nhãn và dữ liệu thiếu

In [ ]:
cac_cot_nhan = [
    "y_recall_success",
    "y_started_within_10m",
    "y_complete_within_30m",
    "y_complete_within_60m",
    "y_snooze",
    "y_forgot",
]

display(df[cac_cot_nhan].describe(include="all").T)

print("\nSố giá trị thiếu theo cột:")
display(df.isna().sum().sort_values(ascending=False).head(15))

## Chuẩn bị hàm huấn luyện mô hình phân loại

Notebook dùng `RandomForestClassifier` để dễ chạy trên máy thông thường.

Khi có dữ liệu thật lớn hơn, có thể thay bằng:
- LightGBM;
- XGBoost;
- mô hình theo chuỗi;
- mô hình chọn hành động thích nghi.

In [ ]:
CAC_COT_BO_QUA = {
    "user_id",
    "occurrence_id",
    "decision_time",
}

TAT_CA_NHAN = {
    "y_recall_success",
    "y_started_within_10m",
    "y_start_delay_minutes",
    "y_complete_within_30m",
    "y_complete_within_60m",
    "y_completion_delay_minutes",
    "y_snooze",
    "y_forgot",
}

def huan_luyen_bo_phan_loai(df, target):
    work = df[df[target].notna()].copy()
    work = work.sort_values("decision_time").reset_index(drop=True)

    feature_cols = [
        c for c in work.columns
        if c not in CAC_COT_BO_QUA
        and c not in TAT_CA_NHAN
    ]

    # Chia dữ liệu theo thời gian, không xáo trộn ngẫu nhiên
    cut = max(1, int(len(work) * 0.7))
    if cut >= len(work):
        cut = len(work) - 1

    train = work.iloc[:cut]
    test = work.iloc[cut:]

    X_train = train[feature_cols]
    y_train = train[target].astype(int)

    X_test = test[feature_cols]
    y_test = test[target].astype(int)

    cat_cols = [
        c for c in feature_cols
        if X_train[c].dtype == "object"
    ]
    num_cols = [
        c for c in feature_cols
        if c not in cat_cols
    ]

    preprocess = ColumnTransformer([
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            num_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            cat_cols
        )
    ])

    model = RandomForestClassifier(
        n_estimators=250,
        max_depth=8,
        random_state=42,
        class_weight="balanced"
    )

    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    print("Target:", target)
    print("Train rows:", len(train))
    print("Test rows:", len(test))

    y_pred = None
    y_prob = None

    if len(test) > 0:
        y_pred = pipe.predict(X_test)

        print(classification_report(
            y_test, y_pred, zero_division=0
        ))

        if hasattr(pipe, "predict_proba"):
            prob = pipe.predict_proba(X_test)
            classes = list(pipe.named_steps["model"].classes_)
            if 1 in classes:
                y_prob = prob[:, classes.index(1)]

        if y_prob is not None and len(set(y_test)) > 1:
            print(
                "ROC-AUC:",
                round(roc_auc_score(y_test, y_prob), 4)
            )

    return {
        "pipeline": pipe,
        "feature_columns": feature_cols,
        "target": target,
        "train_rows": len(train),
        "test_rows": len(test),
        "X_test": X_test,
        "y_test": y_test,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

## Huấn luyện mô hình dự đoán khả năng tự nhớ

In [ ]:
mo_hinh_tu_nho = huan_luyen_bo_phan_loai(
    df,
    "y_recall_success"
)

## Huấn luyện mô hình dự đoán hoàn thành trong 30 phút

In [ ]:
mo_hinh_hoan_thanh = huan_luyen_bo_phan_loai(
    df,
    "y_complete_within_30m"
)

## Đánh giá hai mô hình

Phần này bổ sung bốn nhóm chỉ số trực quan cho cả hai mô hình:

- **Ma trận nhầm lẫn**: cho biết số trường hợp dự đoán đúng/sai theo từng lớp.
- **Độ chính xác (Accuracy)**: tỷ lệ tổng số dự đoán đúng.
- **Độ bao phủ (Recall)**: trong các trường hợp thực sự là lớp dương, mô hình nhận ra được bao nhiêu.
- **Đường cong ROC và ROC-AUC**: đánh giá khả năng phân biệt hai lớp trên nhiều ngưỡng dự đoán.

> Vì dữ liệu mẫu hiện còn nhỏ, các chỉ số chỉ có ý nghĩa kiểm thử pipeline. Khi dùng dữ liệu thật, cần nhiều phiên hơn và tập kiểm thử theo thời gian đủ lớn.

In [ ]:
def danh_gia_mo_hinh_nhi_phan(model_bundle, model_name):
    y_test = model_bundle["y_test"]
    y_pred = model_bundle["y_pred"]
    y_prob = model_bundle["y_prob"]

    if y_pred is None or len(y_test) == 0:
        print(f"{model_name}: chưa có đủ dữ liệu kiểm thử.")
        return None

    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    print("=" * 70)
    print(model_name)
    print("Độ chính xác (Accuracy):", round(accuracy, 4))
    print("Độ bao phủ (Recall):", round(recall, 4))
    print("Ma trận nhầm lẫn:")
    print(cm)

    metrics = {
        "model": model_name,
        "accuracy": accuracy,
        "recall": recall,
        "confusion_matrix": cm,
        "roc_auc": np.nan,
    }

    if y_prob is not None and len(set(y_test)) > 1:
        metrics["roc_auc"] = roc_auc_score(y_test, y_prob)

    return metrics

chi_so_tu_nho = danh_gia_mo_hinh_nhi_phan(
    mo_hinh_tu_nho,
    "Mô hình dự đoán khả năng tự nhớ"
)

chi_so_hoan_thanh = danh_gia_mo_hinh_nhi_phan(
    mo_hinh_hoan_thanh,
    "Mô hình dự đoán hoàn thành trong 30 phút"
)

bang_chi_so = pd.DataFrame([
    {
        "Mô hình": chi_so_tu_nho["model"],
        "Độ chính xác": chi_so_tu_nho["accuracy"],
        "Độ bao phủ": chi_so_tu_nho["recall"],
        "ROC-AUC": chi_so_tu_nho["roc_auc"],
    },
    {
        "Mô hình": chi_so_hoan_thanh["model"],
        "Độ chính xác": chi_so_hoan_thanh["accuracy"],
        "Độ bao phủ": chi_so_hoan_thanh["recall"],
        "ROC-AUC": chi_so_hoan_thanh["roc_auc"],
    },
])

display(bang_chi_so)

### Ma trận nhầm lẫn – mô hình dự đoán khả năng tự nhớ

In [ ]:
cm = chi_so_tu_nho["confusion_matrix"]

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Ma trận nhầm lẫn – Khả năng tự nhớ")
plt.xlabel("Nhãn dự đoán")
plt.ylabel("Nhãn thực tế")
plt.xticks([0, 1], ["Không nhớ", "Tự nhớ"])
plt.yticks([0, 1], ["Không nhớ", "Tự nhớ"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

### Ma trận nhầm lẫn – mô hình dự đoán hoàn thành trong 30 phút

In [ ]:
cm = chi_so_hoan_thanh["confusion_matrix"]

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Ma trận nhầm lẫn – Hoàn thành trong 30 phút")
plt.xlabel("Nhãn dự đoán")
plt.ylabel("Nhãn thực tế")
plt.xticks([0, 1], ["Không hoàn thành", "Hoàn thành"])
plt.yticks([0, 1], ["Không hoàn thành", "Hoàn thành"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

### So sánh độ chính xác của hai mô hình

In [ ]:
bang_do_chinh_xac = bang_chi_so[["Mô hình", "Độ chính xác"]].copy()

plt.figure(figsize=(8, 4))
plt.bar(bang_do_chinh_xac["Mô hình"], bang_do_chinh_xac["Độ chính xác"])
plt.ylim(0, 1)
plt.ylabel("Độ chính xác")
plt.title("Độ chính xác của hai mô hình OneStep")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

### So sánh độ bao phủ của hai mô hình

In [ ]:
bang_do_bao_phu = bang_chi_so[["Mô hình", "Độ bao phủ"]].copy()

plt.figure(figsize=(8, 4))
plt.bar(bang_do_bao_phu["Mô hình"], bang_do_bao_phu["Độ bao phủ"])
plt.ylim(0, 1)
plt.ylabel("Độ bao phủ")
plt.title("Độ bao phủ của hai mô hình OneStep")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

### Đường cong ROC – mô hình dự đoán khả năng tự nhớ

In [ ]:
y_test = mo_hinh_tu_nho["y_test"]
y_prob = mo_hinh_tu_nho["y_prob"]

if y_prob is not None and len(set(y_test)) > 1:
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    auc_value = roc_auc_score(y_test, y_prob)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {auc_value:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Mức ngẫu nhiên")
    plt.xlabel("Tỷ lệ dương giả")
    plt.ylabel("Tỷ lệ dương thật")
    plt.title("Đường cong ROC – Khả năng tự nhớ")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Tập kiểm thử chưa có đủ hai lớp để vẽ ROC cho mô hình tự nhớ.")

### Đường cong ROC – mô hình dự đoán hoàn thành trong 30 phút

In [ ]:
y_test = mo_hinh_hoan_thanh["y_test"]
y_prob = mo_hinh_hoan_thanh["y_prob"]

if y_prob is not None and len(set(y_test)) > 1:
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    auc_value = roc_auc_score(y_test, y_prob)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {auc_value:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Mức ngẫu nhiên")
    plt.xlabel("Tỷ lệ dương giả")
    plt.ylabel("Tỷ lệ dương thật")
    plt.title("Đường cong ROC – Hoàn thành trong 30 phút")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Tập kiểm thử chưa có đủ hai lớp để vẽ ROC cho mô hình hoàn thành.")

### Cách diễn giải các biểu đồ

- **Độ chính xác cao** cho biết mô hình dự đoán đúng nhiều trường hợp nói chung.
- **Độ bao phủ cao** đặc biệt quan trọng khi OneStep muốn tránh bỏ sót các trường hợp người dùng thực sự có thể tự nhớ hoặc thực sự có thể hoàn thành.
- **Ma trận nhầm lẫn** giúp nhìn rõ lỗi loại nào xảy ra nhiều hơn, thay vì chỉ nhìn một con số tổng.
- **ROC-AUC gần 1** cho thấy khả năng phân biệt hai lớp tốt; gần 0,5 tương đương mức ngẫu nhiên.
- Với dữ liệu hành vi cá nhân, nên theo dõi các chỉ số này theo **thời gian và theo từng người dùng**, không chỉ báo cáo một lần trên toàn bộ dữ liệu.

## Lưu mô hình

In [ ]:
joblib.dump(
    mo_hinh_tu_nho,
    "pcbm_recall_model.joblib"
)

joblib.dump(
    mo_hinh_hoan_thanh,
    "pcbm_completion_30m_model.joblib"
)

print("Đã lưu 2 model.")

## Tạo một mẫu đầu vào mới để kiểm thử

Đây là ví dụ của một người dùng có:
- lịch sử hoàn thành khá tốt;
- mức nhắc hiện tại là 1;
- nhiệm vụ học tập vào buổi tối.

In [ ]:
du_lieu_mau = {
    "task_id": "MATH",
    "hour": 19,
    "weekday": 1,
    "is_weekend": 0,
    "xac_dinh_buoi_trong_ngay": "buổi_tối",
    "hour_sin": math.sin(2 * math.pi * 19 / 24),
    "hour_cos": math.cos(2 * math.pi * 19 / 24),

    "task_type": "homework",
    "priority": 2,
    "estimated_minutes": 30,

    "prompt_level": 1,
    "reminder_type": "generic",
    "channel": "buddy",

    "completion_rate_7d": 0.82,
    "snooze_rate_7d": 0.15,
    "forget_rate_7d": 0.08,
    "avg_start_delay_7d": 7.5,

    "same_task_completion_rate": 0.78,

    "event_count_before": 120,
    "history_days": 30,
    "is_cold_start": 0,
}

du_lieu_mau

## Dự đoán với mẫu mới

In [ ]:
def du_doan_xac_suat(model_bundle, sample):
    cols = model_bundle["feature_columns"]
    row = {
        c: sample.get(c, np.nan)
        for c in cols
    }
    X = pd.DataFrame([row], columns=cols)

    pipe = model_bundle["pipeline"]
    proba = pipe.predict_proba(X)

    classes = list(pipe.named_steps["model"].classes_)
    if 1 in classes:
        return float(proba[0][classes.index(1)])
    return float(proba[0][-1])

xac_suat_tu_nho = du_doan_xac_suat(
    mo_hinh_tu_nho,
    du_lieu_mau
)

xac_suat_hoan_thanh_30p = du_doan_xac_suat(
    mo_hinh_hoan_thanh,
    du_lieu_mau
)

print("Xác suất tự nhớ:", round(xac_suat_tu_nho, 3))
print("Xác suất hoàn thành trong 30 phút:", round(xac_suat_hoan_thanh_30p, 3))

## chuyển dự đoán thành gợi ý hỗ trợ

- nếu xác suất tự nhớ cao → giảm một mức hỗ trợ;
- nếu xác suất tự nhớ thấp → tăng một mức hỗ trợ;
- còn lại → giữ nguyên.

In [ ]:
muc_nhac_hien_tai = du_lieu_mau["prompt_level"]

if xac_suat_tu_nho >= 0.80:
    muc_nhac_de_xuat = max(
        0,
        muc_nhac_hien_tai - 1
    )
    ly_do = "Khả năng tự nhớ cao"
elif xac_suat_tu_nho < 0.45:
    muc_nhac_de_xuat = min(
        2,
        muc_nhac_hien_tai + 1
    )
    ly_do = "Khả năng tự nhớ thấp"
else:
    muc_nhac_de_xuat = muc_nhac_hien_tai
    ly_do = "Giữ mức hỗ trợ hiện tại"

khuyen_nghi = {
    "recall_probability": xac_suat_tu_nho,
    "completion_30m_probability": xac_suat_hoan_thanh_30p,
    "muc_nhac_hien_tai": muc_nhac_hien_tai,
    "muc_nhac_de_xuat": muc_nhac_de_xuat,
    "ly_do": ly_do,
}

khuyen_nghi